> **Snap Market model series.** We start from Carl's baseline momentum model and iterate to test it and improve it.
>
> **Notebook 09 — defence in depth.** Combine both ideas: display the direction (volatility-regime momentum) and charge an information margin on top, then test the combined model against the attackers.

# 09 — Defence in depth: guarded volatility-regime momentum

The `guarded_volatility_regime_momentum` model combines the two defences from notebook 08:

- it **displays** the volatility-regime momentum curve (pricing the direction, like `volatility_regime_momentum`), and
- it **adds an information margin** computed from a richer hidden logistic estimate (like `hidden_symmetric_margin` / model 3), widening the spread where that hidden signal is strong.

We compare it to its two components against the three informed attackers, on the common
out-of-sample window.

In [ ]:
import os
import sys

# Move up one level to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# Add the new current directory to the Python path
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from snapmarket.data import load_oracle_prices, load_fast_feed
from snapmarket.features import build_features
from snapmarket.parameters import SharedParameters
from snapmarket.models import build_model
from snapmarket.experiments import common_evaluation_start
from snapmarket.signals import walk_forward_logistic_probability, regime_conditional_probability
from snapmarket.strategies import (
    noise_pool, predictive_bettor, lead_lag_bettor, regime_aware_bettor,
)
from snapmarket.engine import simulate

shared_parameters = SharedParameters()
features = build_features(load_oracle_prices(), shared_parameters)

models = {
    "volatility_regime_momentum": build_model("volatility_regime_momentum", features, shared_parameters),
    "hidden margin (model 3)": build_model("hidden_symmetric_margin", features, shared_parameters),
    "guarded (combined)": build_model("guarded_volatility_regime_momentum", features, shared_parameters),
}
model_labels = list(models)
color_map = dict(zip(model_labels, ["#9ca3af", "#f59e0b", "#2563eb"]))

evaluation_start = common_evaluation_start(models.values())
print(f"common evaluation start: second {evaluation_start:,} (~{evaluation_start / 86_400:.0f} days in)")

## Part 1 — Uninformed house edge

In [ ]:
window_length = 150_000
number_of_windows = 6

rows = []
for label, model in models.items():
    house_pnl = volume = 0.0
    for window_index in range(number_of_windows):
        start = evaluation_start + window_index * window_length
        if start + window_length + shared_parameters.horizon_seconds >= features.number_of_seconds:
            break
        result = simulate(model, features, {"noise": noise_pool()}, start, window_length,
                          seed=1_000 + window_index)
        house_pnl += result.house_pnl
        volume += result.total_volume
    rows.append({"model": label, "house_edge": house_pnl / volume})

house = pd.DataFrame(rows).set_index("model").reindex(model_labels)
print(house.to_string(formatters={"house_edge": "{:+.3%}".format}))

## Part 2 — Informed attackers

In [ ]:
fast_feed = load_fast_feed(expected_length=features.number_of_seconds)
logistic_probability = walk_forward_logistic_probability(features, shared_parameters)
regime_probability = regime_conditional_probability(features, shared_parameters)
informed_attackers = {
    "predictive (logistic)": predictive_bettor(logistic_probability),
    "lead-lag (fast feed)": lead_lag_bettor(features, fast_feed.log_price),
    "regime-aware": regime_aware_bettor(regime_probability),
}

attacker_rows = []
for model_label, model in models.items():
    for attacker_label, bettor in informed_attackers.items():
        attacker_pnl = attacker_stake = 0.0
        for window_index in range(number_of_windows):
            start = evaluation_start + window_index * window_length
            if start + window_length + shared_parameters.horizon_seconds >= features.number_of_seconds:
                break
            result = simulate(model, features, {"pool": noise_pool(), "attacker": bettor},
                              start, window_length, seed=200 + window_index)
            outcome = result.per_bettor["attacker"]
            attacker_pnl += outcome.pnl
            attacker_stake += outcome.stake
        attacker_rows.append({"attacker": attacker_label, "model": model_label,
                              "attacker_edge": attacker_pnl / attacker_stake if attacker_stake else 0.0})

attacker_edge = (pd.DataFrame(attacker_rows)
                 .pivot(index="attacker", columns="model", values="attacker_edge")[model_labels])
print(attacker_edge.to_string(formatters={label: "{:+.2%}".format for label in model_labels}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
attacker_edge.plot(kind="bar", ax=ax, color=color_map)
ax.axhline(0, color="#16a34a", ls="--", label="break-even (beats the house above this)")
ax.axhline(-shared_parameters.house_margin, color="#dc2626", ls=":",
           label=f"uninformed baseline (-vig = {-shared_parameters.house_margin:.1%})")
ax.set_title("Informed attacker edge: components vs guarded (defence in depth)")
ax.set_ylabel("attacker edge"); ax.set_xlabel("")
ax.legend(fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## Recorded results

6 windows of 150,000 seconds, common out-of-sample start ~90 days in.

**Uninformed house edge**

| model | house edge |
|---|---|
| volatility_regime_momentum | +12.91% |
| hidden margin (model 3) | +13.87% |
| guarded (combined) | +13.44% |

**Informed attacker edge** (positive = beats the house)

| attacker | volatility_regime_momentum | hidden margin (model 3) | guarded (combined) |
|---|---|---|---|
| predictive (logistic) | -4.04% | -48.97% | **-100.00%** |
| lead-lag (fast feed) | -17.18% | -18.03% | -17.89% |
| regime-aware | +2.97% | +3.54% | **+2.82%** |

**Read.** Defence in depth works: the guarded model is the best (or tied-best) against every
attacker. It inherits model 3's predictive defence — the information margin is built on the same
logistic the predictive attacker uses, so that attacker is fully priced out (-100%) — and it
inherits the direction pricing, giving the lowest regime-aware edge of the three (+2.82%). Its
uninformed vig (+13.44%) sits above the direction-only model and just below pure model 3. The only
residual leak is the regime-aware attacker (+2.82%), whose finer regime structure neither the
3-tercile display nor the logistic fully captures — the next lever would be a richer volatility
conditioning or more data per regime.

![attacker edge](assets/09_guarded_defence_in_depth_attackers.png)